# 001 · SQLite Basics

> **Standard Library › Persistence** · Python 3.14

Python ships with a complete relational database: the `sqlite3` module embeds the SQLite engine, so you get SQL, indexes and transactions in a single file (or purely in memory) with nothing to install. It is perfect for local tools, caches, tests and small applications.

**In this notebook you will learn:**

- how `sqlite3` maps to ADO.NET / `Microsoft.Data.Sqlite`, and what DB-API 2.0 is
- connecting, creating tables, and running statements with `execute` / `executemany`
- **parameter binding** with `?` and `:name` — and why string formatting is an SQL-injection hole
- reading results with `fetchone`, `fetchall` and by iterating the cursor
- `lastrowid`, `rowcount`, closing connections, and how Python types map to SQLite

## .NET mental model

| ADO.NET / `Microsoft.Data.Sqlite` | Python `sqlite3` |
|---|---|
| `new SqliteConnection("Data Source=:memory:")` + `Open()` | `sqlite3.connect(":memory:")` |
| `SqliteCommand` | `Cursor` (or the `connection.execute(...)` shortcut) |
| `cmd.Parameters.AddWithValue("$id", 1)` | `execute(sql, (1,))` or `execute(sql, {"id": 1})` |
| `ExecuteNonQuery()` → affected rows | `cursor.rowcount` |
| `ExecuteReader()` + `reader.Read()` | iterate the cursor / `fetchone()` |
| `ExecuteScalar()` | `execute(...).fetchone()[0]` |
| `SELECT last_insert_rowid()` | `cursor.lastrowid` |
| `SqliteException` | `sqlite3.Error` and its subclasses |
| ADO.NET interfaces (`DbConnection`, `DbCommand`) | **DB-API 2.0** (PEP 249) |

**DB-API 2.0** is Python's equivalent of the ADO.NET abstractions: a standard shape (`connect()`, `cursor()`, `execute()`, `fetchall()`, `commit()`, `rollback()`, a common exception hierarchy) that every database driver follows. Learn it once with `sqlite3`, and drivers for PostgreSQL (`psycopg`), MySQL, SQL Server (`pyodbc`) feel the same — the main difference is the placeholder style.

In [ ]:
import sqlite3

print(sqlite3.sqlite_version)     # the embedded SQLite engine version
print(sqlite3.apilevel)           # DB-API level "2.0"
print(sqlite3.paramstyle)         # "qmark" -> ? placeholders (named :name also works)

## Connecting

`sqlite3.connect(path)` opens (or creates) a database file. The special name `":memory:"` creates a private, in-memory database that disappears when the connection closes — ideal for demos and tests.

```csharp
// C#
using var conn = new SqliteConnection("Data Source=:memory:");
conn.Open();
```

In [ ]:
conn = sqlite3.connect(":memory:")
print(type(conn).__name__, conn.in_transaction)

## Creating a table and inserting rows

`conn.execute(sql, params)` is a shortcut that creates a cursor, runs the statement and returns the cursor. Use `executescript` for multiple statements in one string (no parameters allowed there).

In [ ]:
conn.executescript("""
    CREATE TABLE product (
        id       INTEGER PRIMARY KEY,   -- alias for SQLite's rowid, auto-assigned
        name     TEXT    NOT NULL UNIQUE,
        price    REAL    NOT NULL,
        in_stock INTEGER NOT NULL DEFAULT 1
    );
""")

cur = conn.execute("INSERT INTO product (name, price) VALUES (?, ?)", ("apple", 0.50))
print(cur.lastrowid, cur.rowcount)   # new id, rows affected

conn.commit()

`executemany` runs the same statement for every parameter tuple — the bulk insert workhorse:

In [ ]:
products = [("pear", 0.75), ("melon", 2.10), ("hammer", 12.00), ("nails", 3.25)]

cur = conn.executemany("INSERT INTO product (name, price) VALUES (?, ?)", products)
print(cur.rowcount)    # total rows inserted
conn.commit()

⚠️ **Gotcha:** don't forget `commit()`. With the default settings, `sqlite3` opens a transaction implicitly before `INSERT`/`UPDATE`/`DELETE`, and closing the connection **without committing discards your changes** — the same as disposing a `SqlTransaction` without `Commit()`. Transactions get a whole notebook *(see Standard Library › Persistence › 002 · SQLite Transactions and Row Factories)*.

## Parameter binding: `?` and `:name`

Always pass values as **parameters** — a tuple for `?` placeholders, or a dict for `:name` placeholders. This is the equivalent of `SqlParameter` / `Parameters.AddWithValue`.

```csharp
// C#
cmd.CommandText = "SELECT * FROM product WHERE price < $max";
cmd.Parameters.AddWithValue("$max", 5.0);
```

In [ ]:
cheap = conn.execute("SELECT name, price FROM product WHERE price < ?", (1.0,)).fetchall()
print(cheap)

named = conn.execute(
    "SELECT name FROM product WHERE price BETWEEN :low AND :high ORDER BY price",
    {"low": 1, "high": 5},
).fetchall()
print(named)

⚠️ **Gotcha:** a single parameter still needs a **tuple** — `(1.0,)` with the trailing comma. Passing `(1.0)` is just a float, and passing a bare string makes every character a separate parameter.

In [ ]:
try:
    conn.execute("SELECT * FROM product WHERE name = ?", "pear")   # str is a sequence of 4 chars!
except sqlite3.ProgrammingError as e:
    print(f"{type(e).__name__}: {e}")

## ⚠️ SQL injection

Never build SQL with f-strings, `%` or `+`. It is exactly as dangerous as string-concatenating SQL in C#: a crafted value changes the meaning of the query.

In [ ]:
user_input = "nothing' OR '1'='1"

unsafe_sql = f"SELECT name FROM product WHERE name = '{user_input}'"   # ❌ DON'T
print(unsafe_sql)
print(conn.execute(unsafe_sql).fetchall())          # returns EVERY row!

safe = conn.execute("SELECT name FROM product WHERE name = ?", (user_input,)).fetchall()  # ✅
print(safe)                                          # [] — treated as a plain value

Placeholders work for **values** only — not for table or column names. If you must choose a column dynamically, validate it against an allow-list:

In [ ]:
def sort_products(column: str):
    if column not in {"name", "price"}:
        raise ValueError(f"cannot sort by {column!r}")
    return conn.execute(f"SELECT name FROM product ORDER BY {column}").fetchall()

print(sort_products("price"))
try:
    sort_products("price; DROP TABLE product")
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## Reading results

A cursor is an iterator over result rows; each row is a **tuple** by default (named access is covered in the next notebook).

| Method | Returns | ADO.NET analogue |
|---|---|---|
| `fetchone()` | next row or `None` | `reader.Read()` + read columns |
| `fetchmany(n)` | up to *n* rows | — |
| `fetchall()` | list of all remaining rows | load into a `DataTable` / list |
| `for row in cursor` | rows lazily | `while (reader.Read())` |

In [ ]:
cur = conn.execute("SELECT id, name, price FROM product ORDER BY id")

print(cur.fetchone())          # first row
print(cur.fetchmany(2))        # next two
for row in cur:                # the rest, lazily
    print(row)
print(cur.fetchone())          # exhausted -> None

"Scalar" queries are just `fetchone()[0]`, and `cursor.description` exposes the column names:

In [ ]:
count = conn.execute("SELECT COUNT(*) FROM product").fetchone()[0]   # ExecuteScalar
print(count)

cur = conn.execute("SELECT name, price FROM product LIMIT 1")
print([column[0] for column in cur.description])

missing = conn.execute("SELECT price FROM product WHERE name = ?", ("kiwi",)).fetchone()
print(missing)                 # None — check before indexing!

## `rowcount` and `lastrowid`

- `rowcount` — rows affected by `INSERT`/`UPDATE`/`DELETE` (and totalled over `executemany`); it is `-1` for `SELECT`, since rows are fetched lazily.
- `lastrowid` — the rowid of the last row inserted by a single `execute` (`INTEGER PRIMARY KEY` columns *are* the rowid).

In [ ]:
cur = conn.execute("UPDATE product SET price = price * 1.1 WHERE price > ?", (2,))
print("updated:", cur.rowcount)

cur = conn.execute("DELETE FROM product WHERE name = ?", ("nails",))
print("deleted:", cur.rowcount)

cur = conn.execute("INSERT INTO product (name, price) VALUES (?, ?)", ("kiwi", 0.30))
print("new id:", cur.lastrowid)   # 5 again: without AUTOINCREMENT, a deleted max id is reused

print(conn.execute("SELECT * FROM product").rowcount)    # -1 for SELECT
conn.commit()

**Tip:** SQLite 3.35+ supports `RETURNING`, which is often neater than `lastrowid`, especially for multiple generated columns:

In [ ]:
row = conn.execute(
    "INSERT INTO product (name, price) VALUES (?, ?) RETURNING id, in_stock", ("plum", 0.40)
).fetchone()
print(row)
conn.commit()

## Type mapping

| Python | SQLite storage class |
|---|---|
| `None` | `NULL` |
| `int` (64-bit range) | `INTEGER` |
| `float` | `REAL` |
| `str` | `TEXT` |
| `bytes` | `BLOB` |
| `bool` | stored as `INTEGER` 0/1 — read back as `int` |
| `datetime`, `Decimal`, ... | not built in — register adapters (next notebook) |

In [ ]:
row = conn.execute(
    "SELECT ?, ?, ?, ?, ?, ?, typeof(?)", (None, 42, 3.14, "text", b"\x00\x01", True, True)
).fetchone()
print(row)

from decimal import Decimal
try:
    conn.execute("SELECT ?", (Decimal("19.99"),))
except sqlite3.ProgrammingError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** SQLite uses **flexible typing** — a column's declared type is only an *affinity*. Unlike SQL Server, `'abc'` can be stored in an `INTEGER` column. Declare tables `STRICT` (SQLite 3.37+) to get real type enforcement.

In [ ]:
conn.execute("CREATE TABLE loose (qty INTEGER)")
conn.execute("INSERT INTO loose VALUES (?), (?)", ("42", "abc"))
print(conn.execute("SELECT qty, typeof(qty) FROM loose").fetchall())   # '42' converted, 'abc' kept

conn.execute("CREATE TABLE strict_qty (qty INTEGER) STRICT")
try:
    conn.execute("INSERT INTO strict_qty VALUES (?)", ("abc",))
except sqlite3.IntegrityError as e:
    print(f"{type(e).__name__}: {e}")

## Errors

All exceptions derive from `sqlite3.Error` (DB-API names, shared by every driver):

| Exception | Typical cause |
|---|---|
| `OperationalError` | SQL syntax error, missing table, locked database |
| `IntegrityError` | `UNIQUE`, `NOT NULL`, `FOREIGN KEY`, `CHECK` violations |
| `ProgrammingError` | wrong number of parameters, unsupported type, closed connection |

In [ ]:
attempts = [
    ("SELECT * FROM missing_table", ()),
    ("INSERT INTO product (name, price) VALUES (?, ?)", ("apple", 1.0)),   # duplicate name
    ("SELECT * FROM product WHERE id = ? AND name = ?", (1,)),              # too few params
]
for sql, params in attempts:
    try:
        conn.execute(sql, params)
    except sqlite3.Error as e:
        print(f"{type(e).__name__}: {e}")

## Closing connections

Call `close()` when you are done (like `Dispose()`). Note that **`with conn:` does *not* close the connection** — it manages a transaction (next notebook). To get `using`-style closing, wrap the connection in `contextlib.closing` *(see Syntax › Error Handling › 003 · Context Managers)*.

```csharp
// C#
using (var conn = new SqliteConnection("Data Source=:memory:")) { conn.Open(); /* ... */ }  // disposed here
```

In [ ]:
from contextlib import closing

with closing(sqlite3.connect(":memory:")) as temp_conn:
    print(temp_conn.execute("SELECT 'hello from sqlite'").fetchone()[0])

try:
    temp_conn.execute("SELECT 1")
except sqlite3.ProgrammingError as e:
    print(f"{type(e).__name__}: {e}")

And a file-based database, kept inside a temporary directory:

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    db_path = Path(tmp) / "shop.db"

    with closing(sqlite3.connect(db_path)) as file_conn:
        file_conn.execute("CREATE TABLE note (text TEXT)")
        file_conn.execute("INSERT INTO note VALUES (?)", ("persisted",))
        file_conn.commit()

    with closing(sqlite3.connect(db_path)) as file_conn:     # reopen: the data is there
        print(file_conn.execute("SELECT text FROM note").fetchall())
    print(db_path.name, db_path.stat().st_size > 0)

In [ ]:
conn.close()    # close the notebook's main connection

## Summary

| Task | C# / ADO.NET | Python `sqlite3` |
|---|---|---|
| Open | `new SqliteConnection(cs).Open()` | `sqlite3.connect(path)` |
| In-memory DB | `Data Source=:memory:` | `":memory:"` |
| Run a statement | `cmd.ExecuteNonQuery()` | `conn.execute(sql, params)` |
| Bulk insert | loop + reuse command | `conn.executemany(sql, rows)` |
| Parameters | `Parameters.AddWithValue("$x", v)` | `(v,)` for `?`, `{"x": v}` for `:x` |
| Read rows | `ExecuteReader()` / `Read()` | `for row in cursor` / `fetchall()` |
| Scalar | `ExecuteScalar()` | `execute(...).fetchone()[0]` |
| Rows affected | return of `ExecuteNonQuery()` | `cursor.rowcount` |
| New id | `last_insert_rowid()` | `cursor.lastrowid` / `RETURNING` |
| Commit | `transaction.Commit()` | `conn.commit()` |
| Dispose | `using var conn = ...` | `with closing(sqlite3.connect(...))` |

## Exercises

1. Create an in-memory `sensor_reading(sensor TEXT, value REAL)` table, insert five readings for two sensors with `executemany`, and print the average value per sensor (`GROUP BY`).
2. Write `find_products(conn, max_price, name_prefix)` that uses **named** parameters and `LIKE` to return matching product names. (Hint: build the pattern `prefix + "%"` in Python and bind it as a value.)
3. Insert a row and print its new id using both `lastrowid` and `RETURNING`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import sqlite3
from contextlib import closing

with closing(sqlite3.connect(":memory:")) as conn:
    # 1 — executemany + GROUP BY
    conn.execute("CREATE TABLE sensor_reading (sensor TEXT, value REAL)")
    conn.executemany("INSERT INTO sensor_reading VALUES (?, ?)",
                     [("t1", 20.0), ("t1", 22.0), ("t2", 18.5), ("t2", 19.5), ("t2", 21.0)])
    for sensor, avg in conn.execute(
            "SELECT sensor, AVG(value) FROM sensor_reading GROUP BY sensor ORDER BY sensor"):
        print(sensor, round(avg, 2))

    # 2 — the wildcard is part of the bound value, never of the SQL text
    conn.execute("CREATE TABLE product (id INTEGER PRIMARY KEY, name TEXT, price REAL)")
    conn.executemany("INSERT INTO product (name, price) VALUES (?, ?)",
                     [("apple", 0.5), ("apricot", 1.2), ("avocado", 2.5), ("pear", 0.7)])

    def find_products(conn, max_price, name_prefix):
        rows = conn.execute(
            "SELECT name FROM product WHERE price <= :max AND name LIKE :pattern ORDER BY name",
            {"max": max_price, "pattern": name_prefix + "%"},
        )
        return [name for (name,) in rows]

    print(find_products(conn, 2.0, "ap"))   # ['apple', 'apricot']

    # 3 — two ways to get the generated key
    cur = conn.execute("INSERT INTO product (name, price) VALUES (?, ?)", ("fig", 0.9))
    print(cur.lastrowid)
    print(conn.execute("INSERT INTO product (name, price) VALUES (?, ?) RETURNING id",
                       ("lime", 0.4)).fetchone()[0])
    conn.commit()
```

</details>